# **Étape 1 : importation des bibliothèques**

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import yaml
from ultralytics import YOLO

import matplotlib.pyplot as plt

import shutil



# **Étape 2 : Visualiser les données**

On ne peut pas entraîner une IA à l'aveugle. On doit d'abord vérifier que nos images et nos étiquettes (les boîtes) se superposent correctement.

In [ ]:
# 1. On définit les chemins exacts d'après ce que tu as trouvé
dossier_images = '/kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed/images/val'
dossier_labels = '/kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed/labels/val'

# On choisit l'image val_30
chemin_img = os.path.join(dossier_images, 'val_30.jpg')
chemin_txt = os.path.join(dossier_labels, 'val_30.txt')

In [ ]:

# 2. On charge l'image
try:
    img = Image.open(chemin_img)
    largeur_img, hauteur_img = img.size

    # 3. On prépare l'affichage
    fig, ax = plt.subplots(1, figsize=(12, 12))
    ax.imshow(img)

    # 4. On lit le fichier texte ligne par ligne pour dessiner les boîtes
    with open(chemin_txt, 'r') as file:
        lignes = file.readlines()
        for ligne in lignes:
            # YOLO format : classe x_centre y_centre largeur hauteur (valeurs entre 0 et 1)
            elements = ligne.strip().split()
            classe, x_centre, y_centre, largeur, hauteur = map(float, elements)

            # On reconvertit les pourcentages en pixels
            w = largeur * largeur_img
            h = hauteur * hauteur_img
            x = (x_centre * largeur_img) - (w / 2)
            y = (y_centre * hauteur_img) - (h / 2)

            # On dessine un rectangle rouge
            rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
            ax.add_patch(rect)

    plt.title(f"Image val_30 avec {len(lignes)} objets détectés")
    plt.axis('off')
    plt.show()

except FileNotFoundError:
    print(f"Oups ! L'image ou le fichier texte n'a pas été trouvé à ce chemin : {chemin_img}")

# **Étape 3 : Installer le cerveau (YOLO)**

Maintenant que l'on a compris nos données, on va préparer l'outil qui va les analyser. On va utiliser la bibliothèque ultralytics. C'est le standard actuel dans l'industrie pour utiliser YOLOv8 (le modèle le plus performant pour ce genre de tâche).


In [ ]:
# Installation de la bibliothèque Ultralytics (qui contient YOLOv8)
#Le petit -q sert juste à dire "quiet" pour ne pas afficher des centaines de lignes de téléchargement
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

# **Étape 4 : Créer la "Carte GPS" pour YOLO (Le fichier YAML)**
Avant de dire à YOLO "Commence à apprendre !", on doit lui donner une carte pour qu'il trouve son chemin dans les dossiers. YOLO ne devine pas tout seul où sont les images d'entraînement (train) et les images de validation (val).

Pour ça, YOLO utilise un petit fichier de configuration qui s'appelle un fichier YAML.
Même s'il y en a déjà un dans la dataset, le plus sûr sur Kaggle est d'en générer un nous-mêmes avec les chemins exacts de ton environnement pour éviter la moindre erreur.


In [ ]:


# On définit le chemin principal vers ton dataset
chemin_base = '/kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed'

# On crée le dictionnaire de configuration pour YOLO
configuration_yolo = {
    'path': chemin_base,               # Chemin racine
    'train': 'images/train',           # Dossier des images pour l'apprentissage
    'val': 'images/val',               # Dossier des images pour l'évaluation pendant l'apprentissage
    'test': 'images/test',             # Dossier des images pour le test final
    'names': {
        0: 'produit'                   # La classe 0 s'appellera "produit"
    }
}

# On sauvegarde ça dans un fichier dataset.yaml
chemin_yaml = '/kaggle/working/dataset.yaml'

with open(chemin_yaml, 'w') as fichier:
    yaml.dump(configuration_yolo, fichier, sort_keys=False)

print(f"Succès ! Le fichier de configuration a été créé ici : {chemin_yaml}")

**Le nettoyage des données (Data Cleaning)**
 dans un vrai projet d'entreprise, les données brutes sont souvent un désastre. Des images corrompues, des étiquettes fausses, des formats incompatibles...
Cependant, rappelle du dataset qu'on a téléchargé sur Kaggle : il s'appelait SKU110K_fixed et sa description disait qu'il était "prêt pour YOLOv5". Quelqu'un a déjà fait le nettoyage pour nous ! L'auteur sur Kaggle a déjà formaté les fichiers texte, vérifié les coordonnées des boîtes et structuré les dossiers. C'est l'avantage d'utiliser un dataset communautaire pour démarrer.

**Le Prétraitement et la Data Augmentation (La consigne de la prof)**
la professeur a écrit : "Présenter clairement le dataset et réaliser un prétraitement adapté (visualisation, nettoyage, redimensionnement, normalisation, data augmentation..."

 on n'a pas oublié cette étape, c'est juste que la bibliothèque Ultralytics YOLO le fait automatiquement sous le capot pendant l'entraînement !

Redimensionnement et Normalisation : YOLO va automatiquement redimensionner toutes tes images (par exemple en 640x640 pixels) et diviser les valeurs des pixels pour qu'elles soient entre 0 et 

Data Augmentation : Pendant qu'il s'entraîne, YOLO va lui-même tourner les images, changer la luminosité, faire des zooms aléatoires pour que le modèle devienne robuste.

# Étape 5 : Le "Sanity Check" (Le Mini-Entraînement)
Puisque tout est prêt et que YOLO gère le prétraitement, on va lancer ce qu'on appelle un Sanity Check (un test de bon sens). On va demander au modèle de s'entraîner sur un tout petit nombre d'images (juste 1 seule époque) pour s'assurer que notre fichier YAML est bon, que le GPU chauffe, et que le code ne plante pas au bout de 10 minutes

In [ ]:


model = YOLO('yolov8n.pt') 

results = model.train(
    data='/kaggle/working/dataset.yaml',
    epochs=1,          
    imgsz=640,
    batch=16,
    project='SKU_Detection_Projet',
    name='entrainement_v1'
)

# Étape 6 :entrainement final

In [ ]:

model = YOLO('yolov8n.pt') 

results = model.train(
    data='/kaggle/working/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    verbose=False,      # <--- Désactive les logs détaillés par époque
    plots=True,         # <--- Garde quand même la génération des graphiques en fichiers
    project='SKU_Detection_Projet',
    name='entrainement_final'
)

# Etape 7: Faire un test sur une image de test

Dans cette étape, nous effectuons un test sur une image afin de vérifier le bon fonctionnement du modèle. Cette phase permet d’évaluer les performances du système en conditions réelles et d’observer la précision des résultats obtenus. Elle est essentielle pour détecter d’éventuelles erreurs et améliorer la qualité du modèle.

In [ ]:


# 1. On charge TON modèle que tu viens d'entraîner
model_path = '/kaggle/working/runs/detect/SKU_Detection_Projet/entrainement_v1/weights/best.pt'
mon_ia = YOLO(model_path)

# 2. On prend une image du dossier "test" (données inconnues pour l'IA)
image_test = '/kaggle/input/datasets/thedatasith/sku110k-annotations/SKU110K_fixed/images/test/test_2.jpg'

# 3. L'IA fait sa prédiction
results = mon_ia.predict(source=image_test, conf=0.25) # conf=0.25 : on affiche si l'IA est sûre à 25%

# 4. On affiche le résultat visuel
res_plotted = results[0].plot(labels=False,conf=False) # Dessine les boîtes sur l'image
plt.figure(figsize=(15, 15))
plt.imshow(res_plotted)
plt.title("Résultat de ma première détection automatique !")
plt.axis('off')
plt.show()

# 5. On compte les produits
nombre_produits = len(results[0].boxes)
print(f"L'IA a détecté {nombre_produits} produits sur cette étagère !")

# Étape 8 : sauvegarde des résultats

In [ ]:

# On compresse le dossier des résultats pour pouvoir le télécharger facilement
shutil.make_archive('mes_resultats_yolo', 'zip', '/kaggle/working/runs/detect/SKU_Detection_Projet/entrainement_final')

print("Ton fichier ZIP est prêt ! Regarde dans le panneau 'Output' à droite pour le télécharger.")